# IAD Pipeline
Anomaly detection pipeline using FiftyOne, Weights & Biases, and the IAD framework.

## 1. Environment Setup
Configure database URI and API keys.

In [1]:
import os
import sys
import warnings
import yaml
# Set BEFORE any fiftyone imports
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost"
os.environ["WANDB_API_KEY"] = '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'
sys.path.append("..")

# Now safe to import
import wandb
import logging
from pathlib import Path
from manager import AnomalyDetectionManager as ADM
from manager import DatasetSession as DS

wandb.login()

/Users/dapo/Documents/Code/IAD--Python-/.venv_py312/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
wandb: Currently logged in as: daniel-pommer (daniel-pommer-technische-hochschule-n-rnberg-georg-simon-ohm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 2. Configuration
Set your run parameters here before executing the pipeline.

In [2]:
logger              = logging.getLogger("logger")

productName         = "cable"
datasetName         = "singleImageTest"
split               = ("pred",)

# datasetDir          = Path("../datasets/")
datasetDir = Path(os.getcwd())
configDir           = Path("../configs/")
outputPath          = Path("../results/")
productConfigPath   = Path(f"Products/{productName}.yaml")
productConfigPath=Path(configDir/productConfigPath)

manager, productDescription = ADM.loadProduct(productConfigPath=productConfigPath, outputPath=outputPath, configDir=configDir)
# datasetSession = DS.loadDatasetFromDisk(datasetDir/datasetName, datasetName, split=split)
# datasetSession.select_category(productName)
manager.adjustPaths(datasetName=datasetName, category=productName, adjustCheckpoints=False) # We want to use the checkpoints of the training dataset not the new predictions dataset (does not have checkpoints)
print(f"Output path: {manager.outputPath}")
print(f"Checkpoint path: {manager.ckptDir}")

INFO: Model Patchcore loaded: {'model': {'class_path': 'Patchcore', 'init_args': {'backbone': 'resnet18', 'layers': ['layer2', 'layer3'], 'pre_trained': True, 'coreset_sampling_ratio': 0.1, 'num_neighbors': 9, 'pre_processor': PreProcessor(), 'post_processor': AOIPostProcessor(
  (_image_threshold_metric): F1AdaptiveThreshold()
  (_pixel_threshold_metric): F1AdaptiveThreshold()
  (_image_min_max_metric): MinMax()
  (_pixel_min_max_metric): MinMax()
), 'visualizer': False, 'evaluator': Evaluator(
  (val_metrics): ModuleList(
    (0): AUROC()
  )
  (test_metrics): ModuleList(
    (0): AUROC()
    (1): F1Score()
    (2): AUPR()
  )
)}}}
INFO: Initializing Patchcore model.


/Users/dapo/Documents/Code/IAD--Python-/.venv_py312/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'pre_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['pre_processor'])`.
/Users/dapo/Documents/Code/IAD--Python-/.venv_py312/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'post_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['post_processor'])`.
/Users/dapo/Documents/Code/IAD--Python-/.venv_py312/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'evaluator' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['evaluator'])`.


INFO: Loading pretrained weights from Hugging Face hub (timm/resnet18.a1_in1k)
INFO: [timm/resnet18.a1_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
INFO: Missing keys (fc.weight, fc.bias) discovered while loading pretrained weights. This is expected if model is being adapted.
INFO: Model Patchcore(
  (pre_processor): PreProcessor()
  (post_processor): AOIPostProcessor(
    (_image_threshold_metric): F1AdaptiveThreshold()
    (_pixel_threshold_metric): F1AdaptiveThreshold()
    (_image_min_max_metric): MinMax()
    (_pixel_min_max_metric): MinMax()
  )
  (evaluator): Evaluator(
    (val_metrics): ModuleList(
      (0): AUROC()
    )
    (test_metrics): ModuleList(
      (0): AUROC()
      (1): F1Score()
      (2): AUPR()
    )
  )
  (model): PatchcoreModel(
    (feature_extractor): TimmFeatureExtractor(
      (feature_extractor): FeatureListNet(
        (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padd

## 3. Inspect Dataset

In [3]:
# datasetSession.launchSession()
print(datasetDir/productName)

/Users/dapo/Documents/Code/IAD--Python-/src/AnomalyDetection/src/cable


## 4. Prediction

In [4]:
if manager.ckptPath is not None:
    if not manager.isTilingSetup:
        manager.loadCheckpoint(manager.ckptPath, f"{manager.modelName}")
        print("Loaded Checkpoint")
    if not manager.isTilingSetup:
        manager.setupTiling(configDir / "Tiling" / "TiledEnsemblePred.yaml")
        print("Setup tiling")
    if manager.inferencerPath is not None:
        print(f"Inferencing Singular Image at: {datasetDir/datasetName/productName/"pred"}")
        manager.inferenceSingleImage(
            # imagePath=datasetDir/datasetName/productName,
            datasetName=datasetName,
            datasetDir=datasetDir,
            category=productName,
            inferenceConfigPath=manager.inferencerPath,
            resultsDir=manager.outputPath,
            tiling=manager.isTilingSetup,
            ckptPath=manager.ckptDir,
            trainingDir=productDescription["model"]["trainingDir"])
    print(manager.FO_Dataset.first()["pred_anomaly_Patchcore"]["label"])


Inferencing Singular Image at: /Users/dapo/Documents/Code/IAD--Python-/src/AnomalyDetection/src/singleImageTest/cable/pred
 100% |█████████████████████| 1/1 [9.0ms elapsed, 0s remaining, 126.2 samples/s] 
INFO:  100% |█████████████████████| 1/1 [9.0ms elapsed, 0s remaining, 126.2 samples/s] 
INFO: There are 1 images in the singleImageTest dataset.
INFO: There are 1 categorie(s) in the singleImageTest dataset.
INFO: ['cable']
INFO: Set dataset before adjusting path!
INFO: Running tiled ensemble pred pipeline.
INFO: Setup tiling based on ../configs/Tiling/TiledEnsemble.yaml
INFO: Root directory for Eval Pipeline: ../results/singleImageTest/cable/Patchcore/tiled
INFO: Checkpoint directory: ../results/MVTecADShort/cable/Patchcore/tiled/checkpoints
INFO: Stats directory: ../results/MVTecADShort/cable/Patchcore/tiled
INFO: ckptPath: ../results/MVTecADShort/cable/Patchcore/tiled/checkpoints
INFO: Setting up runners
INFO: Checkpoint path: ../results/MVTecADShort/cable/Patchcore/tiled/checkpoin

Predict: 0it [00:00, ?it/s]

INFO: Tiled ensemble predicting started using Using ckpt_pathtest data.
INFO: Initializing Patchcore model.
INFO: Loading pretrained weights from Hugging Face hub (timm/resnet18.a1_in1k)
INFO: [timm/resnet18.a1_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
INFO: Missing keys (fc.weight, fc.bias) discovered while loading pretrained weights. This is expected if model is being adapted.
INFO: Loading the model.
INFO: Initializing Patchcore model.
INFO: Loading pretrained weights from Hugging Face hub (timm/resnet18.a1_in1k)


/Users/dapo/Documents/Code/IAD--Python-/.venv_py312/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'pre_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['pre_processor'])`.
/Users/dapo/Documents/Code/IAD--Python-/.venv_py312/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'post_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['post_processor'])`.
/Users/dapo/Documents/Code/IAD--Python-/.venv_py312/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'evaluator' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['evaluator'])`.


INFO: [timm/resnet18.a1_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
INFO: Missing keys (fc.weight, fc.bias) discovered while loading pretrained weights. This is expected if model is being adapted.
INFO: Loading checkpoint from ckpt_path: ../results/MVTecADShort/cable/Patchcore/tiled/checkpoints/model0_0.ckpt. No Model from previous training job available.
INFO: Dataset for dataloader: <anomalib.data.predict.PredictDataset object at 0x14c863950>
INFO: Start of predicting for tile at position (0, 0),


...

INFO: Job Visualize completed successfully.
INFO: Running job 51Visualize



51Visualize: 0it [00:00, ?it/s]

INFO: Starting visualisation for Fiftyone.


51 Visualisation: 100%|██████████| 1/1 [00:00<00:00, 30.86it/s]
51Visualize: 1it [00:00, 18.23it/s]

INFO: Job 51Visualize completed successfully.
anomaly


In [5]:
manager.launchSession()


Connected to FiftyOne on port 5151 at localhost.
If you are not connecting to a remote session, you may need to start a new session and specify a port
INFO: Connected to FiftyOne on port 5151 at localhost.
If you are not connecting to a remote session, you may need to start a new session and specify a port


INFO: DatasetView (category selection) currently not supported. Shown dataset is the entire dataset.
INFO: Session addess and port: localhost:5151
